# Load packages

In [1]:
import duckdb
from pathlib import Path
import boto3

# Download and reconstruct the parquet file

In [21]:
bucket = "2006782_flickr_data_segments"

prefix = (
    "flickhistory_nu_full_v2.parquet/"
    "1734371083.332184643/"
    "46600185740/"
)

output_file = "reconstructed.parquet"

# Create a boto3 S3 client
s3_client = boto3.client(
    "s3",
    endpoint_url="https://a3s.fi",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

parts = [f"{i:08d}" for i in range(9)]

with open(output_file, "wb") as outfile:

    for part in parts:
        key = prefix + part

        print("Downloading:", key)

        obj = s3_client.get_object(
            Bucket=bucket,
            Key=key
        )

        while True:
            chunk = obj["Body"].read(1024 * 1024)

            if not chunk:
                break

            outfile.write(chunk)

print("Done.")

Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000000
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000001
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000002
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000003
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000004
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000005
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000006
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000007
Downloading: flickhistory_nu_full_v2.parquet/1734371083.332184643/46600185740/00000008
Done.


In [1]:
with open("reconstructed.parquet", "rb") as f:
    print("START:", f.read(4))

with open("reconstructed.parquet", "rb") as f:
    f.seek(-4, 2)
    print("END:", f.read(4))

START: b'PAR1'
END: b'PAR1'


# Check data with duckdb

In [2]:
con = duckdb.connect()

In [3]:
con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

schema = con.execute("""
DESCRIBE
SELECT *
FROM read_parquet('reconstructed.parquet')
""").df()

print(schema)

    column_name                                        column_type null   key  \
0            id                                             BIGINT  YES  None   
1        server                                            INTEGER  YES  None   
2        secret                                               BLOB  YES  None   
3         title                                            VARCHAR  YES  None   
4   description                                            VARCHAR  YES  None   
5    date_taken                           TIMESTAMP WITH TIME ZONE  YES  None   
6   date_posted                           TIMESTAMP WITH TIME ZONE  YES  None   
7     photo_url                                            VARCHAR  YES  None   
8      page_url                                            VARCHAR  YES  None   
9       user_id                                             BIGINT  YES  None   
10    user_farm                                           SMALLINT  YES  None   
11         tags             

In [4]:
con = duckdb.connect()

df = con.execute("""
SELECT *
FROM read_parquet('reconstructed.parquet')
LIMIT 10
""").df()

print(df)

            id  server                     secret  \
0   8442912086    8215    [58, 246, 87, 103, 168]   
1   8442913778    8048      [5, 73, 133, 64, 237]   
2  21716591396     675   [83, 232, 200, 168, 110]   
3  24654704900    1467  [180, 223, 184, 143, 195]   
4   4368968686    4013       [7, 170, 25, 228, 4]   
5   4402540834    2707  [116, 125, 156, 150, 176]   
6   4404302895    2677       [91, 2, 38, 155, 45]   
7   4404303045    4053  [170, 166, 191, 213, 101]   
8   4404303539    4013      [64, 219, 56, 77, 27]   
9   4404303811    2742  [171, 102, 133, 186, 187]   

                               title  \
0  Yuri fussing though she can't see   
1                       Family pics!   
2                 Sea Lion's Arrival   
3      One Sea Lion & 100 plus Seals   
4                    Sicilian Scraps   
5              The sound of silence.   
6           Market Estate, Islington   
7                 North Coast Sunset   
8                      Under the Tub   
9      Near the 

# Filter data in state-owned lands

In [3]:
# Enable spatial extension
con.execute("INSTALL spatial;")
con.execute("LOAD spatial;")

parquet_path = "reconstructed.parquet"
land_path = "../../../data/boundaries/MH kiinteistot.gpkg"

In [ ]:
import duckdb
import geopandas as gpd
from shapely import wkt

# --- STEP 1: Prepare the Mask ---
# Load Finnish lands (EPSG:3067)
finland_lands = gpd.read_file(land_path)

# Get the bounding box in EPSG:4326 for the "rough cut"
# We reproject the lands just to find the min/max lat/lon
lands_4326 = finland_lands.to_crs("EPSG:4326")
minx, miny, maxx, maxy = lands_4326.total_bounds

# --- STEP 2: The "Rough Cut" (DuckDB) ---
# Connect to DuckDB and load spatial extension
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

print(f"Filtering global 43GB file using BBox: {minx, miny, maxx, maxy}...")

# Filter the 43GB file down to a small "Finland-area" subset
# We convert the 'geom' column (WKB) to a format DuckDB understands
query = f"""
SELECT * 
FROM read_parquet('{parquet_path}')
WHERE 
    (geom_bbox.xmin >= {minx} AND geom_bbox.xmax <= {maxx}) AND 
    (geom_bbox.ymin >= {miny} AND geom_bbox.ymax <= {maxy})
"""
# This returns a standard pandas DataFrame of only the points near Finland
subset_df = con.execute(query).df()

# --- STEP 3: Precise Intersection (GeoPandas) ---
print(f"Rough cut finished. Processing {len(subset_df)} candidate points...")

# Convert subset to a GeoDataFrame (EPSG:4326)
# Assuming 'geom' is WKB bytes; if it's a list of ints, we convert to bytes first
gdf_subset = gpd.GeoDataFrame(
    subset_df, 
    geometry=gpd.GeoSeries.from_wkb(subset_df['geom'].apply(bytes)),
    crs="EPSG:4326"
)

# REPROJECT everything to EPSG:3067 for high-precision Finnish land matching
gdf_subset_projected = gdf_subset.to_crs("EPSG:3067")
# finland_lands is already 3067

# Perform the precise spatial join
# 'inner' join keeps only points that fall inside a state-owned land polygon
final_filtered_data = gpd.sjoin(
    gdf_subset_projected, 
    finland_lands[['geometry']], 
    how="inner", 
    predicate="intersects"
)

# --- STEP 4: Save Result ---
print(f"Final count: {len(final_filtered_data)} points found in state-owned lands.")
final_filtered_data.to_parquet("../../../data/Flickr/flickr_finland_precise_all.parquet")

Filtering global 43GB file using BBox: (np.float64(20.16581880847455), np.float64(59.47564319918726), np.float64(31.586703436911268), np.float64(70.08972719949732))...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rough cut finished. Processing 1721938 candidate points...
Final count: 62499 points found in state-owned lands.


In [ ]:
# Assuming final_filtered_data is your GeoDataFrame from the previous step

# Save to GeoPackage
# 'driver="GPKG"' is explicitly required to ensure the correct format
# 'layer' allows you to name the table inside the SQLite-based gpkg file
final_filtered_data.to_file(
    "../../../data/Flickr/flickr_finland_state_lands_all.gpkg", 
    driver="GPKG", 
    layer="filtered_flickr_points"
)

print("Export complete: flickr_finland_state_lands_all.gpkg")

Export complete: flickr_finland_state_lands.gpkg
